In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import KFold
from collections import Counter

In [ ]:
TRAIN_PATH  = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SAMPLE_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"
 
LABELS = ["A", "B", "C", "D", "E"]
L2I = {l: i for i, l in enumerate(LABELS)}

In [ ]:
print("Loading data")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
sample   = pd.read_csv(SAMPLE_PATH)
print("Train:", train_df.shape, " Test:", test_df.shape)
ANSWER_COL = "answer" if "answer" in train_df.columns else train_df.columns[-1]
print("Answer column:", ANSWER_COL)
print("Sample submission columns:", list(sample.columns))

In [ ]:
def full_text(row):
    return str(row["prompt"]) + " " + " ".join(str(row[c]) for c in LABELS)

def heuristic_scores(row):

    prompt  = str(row["prompt"])
    options = [str(row[c]) for c in LABELS]
    lens    = np.array([len(o) for o in options], dtype=float)
    len_s   = lens / (lens.sum() + 1e-9)
    wcnt    = np.array([len(o.split()) for o in options], dtype=float)
    wc_s    = wcnt / (wcnt.sum() + 1e-9)
    try:
        v    = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit_transform([prompt] + options)
        ts   = cosine_similarity(v[0:1], v[1:])[0]
        tf_s = ts / (ts.sum() + 1e-9)
    except Exception:
        tf_s = np.full(5, 0.2)
    if np.std(lens) >= 20:
        s = 0.85 * len_s + 0.03 * tf_s + 0.12 * wc_s
    else:
        s = 0.60 * len_s + 0.30 * tf_s + 0.10 * wc_s
    ssum = s.sum()
    return s / ssum if ssum > 0 else np.full(5, 0.2)

In [ ]:
print("Building word + char TF-IDF index")
train_texts = [full_text(r) for _, r in train_df.iterrows()]
test_texts  = [full_text(r) for _, r in test_df.iterrows()]

word_vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=200000)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                           sublinear_tf=True, max_features=120000)
word_vec.fit(train_texts + test_texts)
char_vec.fit(train_texts + test_texts)

Xw_tr = word_vec.transform(train_texts).astype(np.float32)
Xc_tr = char_vec.transform(train_texts).astype(np.float32)
Xw_te = word_vec.transform(test_texts).astype(np.float32)
Xc_te = char_vec.transform(test_texts).astype(np.float32)

train_label_idx = np.array([L2I[a] for a in train_df[ANSWER_COL]])
train_heur = [heuristic_scores(r) for _, r in train_df.iterrows()]
test_heur  = [heuristic_scores(r) for _, r in test_df.iterrows()]

WMIX = 0.6  # weight on word-sim vs char-sim in the combined retrieval score
def combined_sim(qw, qc, rw, rc):
    return WMIX * cosine_similarity(qw, rw) + (1 - WMIX) * cosine_similarity(qc, rc)
print("Done.")

In [ ]:
def predict_baseline(sim_word, label_idx_ref, row):

    order = np.argsort(sim_word)[::-1][:30]
    votes = np.zeros(5); tw = 0.0
    for j in order:
        s = sim_word[j]
        if s < 0.80: break
        votes[label_idx_ref[j]] += s ** 3; tw += s ** 3
    h = heuristic_scores(row)
    if tw <= 0:
        return [int(x) for x in np.argsort(h)[::-1][:3]]
    r1 = int(np.argmax(votes))
    hh = h.copy(); hh[r1] = -1; r2 = int(np.argmax(hh)); hh[r2] = -1; r3 = int(np.argmax(hh))
    return [r1, r2, r3]

def predict_v3(sim_comb, sim_word, label_idx_ref, heur,
               p, top_k, base_thr, recall_thr, beta):
 
    # baseline word-sim vote (identical to original: thr=0.80, power=3)
    ow = np.argsort(sim_word)[::-1][:30]
    bvotes = np.zeros(5)
    for j in ow:
        s = sim_word[j]
        if s < base_thr: break
        bvotes[label_idx_ref[j]] += s ** 3
    # combined (word+char) votes for recovery + secondary ranking
    oc = np.argsort(sim_comb)[::-1][:top_k]
    cvotes = np.zeros(5)
    for j in oc:
        s = sim_comb[j]
        if s < recall_thr: break
        cvotes[label_idx_ref[j]] += s ** p

    if bvotes.sum() > 0:
        rank1  = int(np.argmax(bvotes))                       # IDENTICAL to baseline
        secsrc = cvotes if cvotes.sum() > 0 else bvotes
    elif cvotes.sum() > 0:
        rank1  = int(np.argmax(cvotes))                       # recovered match
        secsrc = cvotes
    else:
        return [int(x) for x in np.argsort(heur)[::-1][:3]]   # baseline fallback

    v   = secsrc / secsrc.sum()
    sec = beta * v + (1 - beta) * heur
    sec[rank1] = -1e9
    rank2 = int(np.argmax(sec)); sec[rank2] = -1e9
    rank3 = int(np.argmax(sec))
    return [rank1, rank2, rank3]

def map3(true_idx, pred_lists):
    tot = 0.0
    for t, preds in zip(true_idx, pred_lists):
        sc = 0.0
        for i, pi in enumerate(preds[:3]):
            if pi == t:
                sc = 1.0 / (i + 1); break
        tot += sc
    return tot / len(true_idx)

In [ ]:
print("Cross-validating (baseline vs new) and tuning the blend...")
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# precompute per-fold similarity matrices once (configs only re-score, no re-vectorize)
fold_cache = []
for tr_idx, va_idx in kf.split(range(len(train_df))):
    sim_c = combined_sim(Xw_tr[va_idx], Xc_tr[va_idx], Xw_tr[tr_idx], Xc_tr[tr_idx])
    sim_w = cosine_similarity(Xw_tr[va_idx], Xw_tr[tr_idx])
    fold_cache.append((
        sim_c, sim_w, train_label_idx[tr_idx],
        [train_heur[i] for i in va_idx],
        [train_label_idx[i] for i in va_idx],
        [train_df.iloc[i] for i in va_idx],
    ))

# baseline CV
b_scores = []
for sc, sw, lab, heur, true, rows in fold_cache:
    preds = [predict_baseline(sw[r], lab, rows[r]) for r in range(sw.shape[0])]
    b_scores.append(map3(true, preds))
baseline_cv = float(np.mean(b_scores))

# tune new model
grid = [(p, k, 0.80, rt, be)
        for p in (3, 4) for k in (30, 50)
        for rt in (0.30, 0.40, 0.50) for be in (0.5, 0.7, 0.9)]
best = None
for cfg in grid:
    s = []
    for sc, sw, lab, heur, true, rows in fold_cache:
        preds = [predict_v3(sc[r], sw[r], lab, heur[r], *cfg) for r in range(sc.shape[0])]
        s.append(map3(true, preds))
    m = float(np.mean(s))
    if best is None or m > best[0]:
        best = (m, cfg)
new_cv, best_cfg = best

print(f"Baseline CV MAP@3 : {baseline_cv:.4f}")
print(f"New      CV MAP@3 : {new_cv:.4f}   (config p,top_k,base_thr,recall_thr,beta = {best_cfg})")
print(f"CV improvement    : {new_cv - baseline_cv:+.4f}")

In [ ]:
print("\nGenerating test predictions with the CV-selected model...")
sim_c_te = combined_sim(Xw_te, Xc_te, Xw_tr, Xc_tr)
sim_w_te = cosine_similarity(Xw_te, Xw_tr)

use_new = new_cv >= baseline_cv  # safety: never ship something worse on CV than baseline
print("Shipping:", "NEW model" if use_new else "BASELINE (safety fallback)")

pred_idx = []
for r in range(sim_c_te.shape[0]):
    if use_new:
        idx3 = predict_v3(sim_c_te[r], sim_w_te[r], train_label_idx, test_heur[r], *best_cfg)
    else:
        idx3 = predict_baseline(sim_w_te[r], train_label_idx, test_df.iloc[r])
    pred_idx.append(idx3)

pred_str = [" ".join(LABELS[i] for i in p) for p in pred_idx]

In [ ]:
id_col, pred_col = sample.columns[0], sample.columns[1]
submission = pd.DataFrame({id_col: test_df["id"].values, pred_col: pred_str})

# align to sample order if ids match
try:
    submission = sample[[id_col]].merge(submission, on=id_col, how="left")
except Exception:
    pass

submission.to_csv(OUTPUT_PATH, index=False)

all_p = [p.split() for p in submission[pred_col]]
print("Rank-1 dist :", dict(Counter(p[0] for p in all_p)))
print("Rank-2 dist :", dict(Counter(p[1] for p in all_p)))
print("Rank-3 dist :", dict(Counter(p[2] for p in all_p)))
print(f"\nSaved to {OUTPUT_PATH}")
print("Total rows  :", len(submission))
print(submission.head(10).to_string(index=False))

In [3]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [4]:
# print(train_df.shape)
# print(test_df.shape)

(2000, 8)
(500, 7)


In [7]:
# print(train_df.columns.tolist())
# print(test_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
